# Deployment lifecycle

A practical example of managing the lifecycle of an existing MongoDB Atlas
Local deployment: list deployments, retrieve one, stop and restart it, pause
and resume it, and inspect its logs.

Requires a Docker daemon on the machine running this kernel.

In [1]:
%pip install atlas-local-lib-py


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## List the existing deployments

`list()` returns every local Atlas deployment on this machine, whatever its
state, including ones created outside this library.

In [2]:
from atlas_local import LocalDeployment

for existing in LocalDeployment.list():
    print(f"{existing.name:<30} {existing.state}")

probe-g                        exited
notebook-demo                  exited
probe-f                        exited
probe-a                        exited
probe-e                        exited
probe-d                        exited
probe-c                        exited


## Get the deployment to work with

`get_or_create` keeps this notebook re-runnable: it returns the deployment if
it is already there, and creates it the first time.

`LocalDeployment.get(NAME)` retrieves the deployment with the specified name and raises `GetDeploymentError` if no matching deployment exists.

In [9]:
NAME = "lifecycle-demo"

deployment = LocalDeployment.get_or_create(name=NAME)

print(deployment.name, deployment.state)

lifecycle-demo running


## Stop and start a deployment

Stopping shuts down the deployment and releases its published host port, while preserving its data. Starting it again brings it back. If no fixed host port was specified when the deployment was created, Docker assigns a new available port.

The `deployment` object is a snapshot of the deployment at the time it was retrieved, so it does not update automatically. Call `get()` again to retrieve its current state.


In [ ]:
deployment.stop()

print("State before refreshing:", deployment.state)

refreshed_deployment = LocalDeployment.get(NAME)
print("State after refreshing:", refreshed_deployment.state)

deployment.start()

refreshed_deployment = LocalDeployment.get(NAME)
print("State after starting:", refreshed_deployment.state)

State before refreshing: running
State after refreshing: exited
State after starting: running


## Pause and resume it

Pausing freezes the running processes instead of shutting them down, so
resuming is immediate. It is a different axis from stop/start, and the two do
not mix: a paused deployment has to be resumed with `unpause()`, and calling
`start()` on it fails.

In [18]:
deployment.pause()
print("after pause:  ", LocalDeployment.get(NAME).state)

deployment.unpause()
print("after unpause:", LocalDeployment.get(NAME).state)

after pause:   paused
after unpause: running


Every lifecycle method is also available as a static method taking a name or a
container ID, which is useful when there is no object at hand:

In [19]:
LocalDeployment.stop_deployment(NAME)
print("after stop_deployment: ", LocalDeployment.get(NAME).state)

LocalDeployment.start_deployment(NAME)
print("after start_deployment:", LocalDeployment.get(NAME).state)

after stop_deployment:  exited
after start_deployment: running


## Inspect the logs

`logs()` returns the container output as a list of lines, most recent last.
Without `tail` it returns everything, which is a lot for a deployment that has
been running for a while.

In [15]:
for line in deployment.logs(tail=5, timestamps=True):
    print(line.strip())

2026-08-17T09:37:11.588020799Z {"t":{"$date":"2026-08-17T09:37:11.587+00:00"},"s":"I",  "c":"NETWORK",  "id":6788700, "ctx":"conn84","msg":"Received first command on ingress connection since session start or auth handshake","attr":{"elapsedMillis":0}}
2026-08-17T09:37:11.588995674Z {"t":{"$date":"2026-08-17T09:37:11.588+00:00"},"s":"I",  "c":"-",        "id":20883,   "ctx":"conn83","msg":"Interrupted operation as its client disconnected","attr":{"opId":117761}}
2026-08-17T09:37:11.589006007Z {"t":{"$date":"2026-08-17T09:37:11.588+00:00"},"s":"I",  "c":"NETWORK",  "id":22944,   "ctx":"conn84","msg":"Connection ended","attr":{"remote":"172.17.0.2:44798","isLoadBalanced":false,"isProxyUnixSock":false,"isPriority":false,"uuid":{"uuid":{"$uuid":"6e629dda-1104-4961-9618-6049a94a7d0b"}},"connectionId":84,"local":"172.17.0.2:27017","connectionCount":22}}
2026-08-17T09:37:11.589008591Z {"t":{"$date":"2026-08-17T09:37:11.588+00:00"},"s":"I",  "c":"NETWORK",  "id":22944,   "ctx":"conn85","msg":"C

It can also be narrowed down to a time range, to see only what happened after
the restarts above:

In [16]:
from datetime import datetime, timedelta, timezone

since = datetime.now(timezone.utc) - timedelta(minutes=1)
recent = deployment.logs(since=since)

print(f"{len(recent)} lines in the last minute")

127 lines in the last minute


## Clean up

Deleting removes the container and its data. Skip this cell to keep the
deployment for the next run.

In [20]:
deployment.delete()

print(NAME in [existing.name for existing in LocalDeployment.list()])

False
